In [ ]:
# pip install transformers datasets tokenizers accelerate evaluate

import os
import torch
import pandas as pd
from datasets import Dataset
import evaluate
from transformers import (
    Trainer, TrainingArguments, DataCollatorForSeq2Seq,
    T5Config, T5ForConditionalGeneration, PreTrainedTokenizerFast
)

# ========== CONFIGURATION ==========
CURRENT_BATCH = 54 # 🔁 Change this for each batch (1, 2, 3, ..., 30)
CSV_PATHS = [f"./datasets/python_batch ({CURRENT_BATCH}).csv"]
PREV_MODEL_DIR = f"scratch_llm_model_batch_{CURRENT_BATCH - 1}" if CURRENT_BATCH > 1 else None
MODEL_DIR = f"scratch_llm_model_batch_{CURRENT_BATCH}"
TOKENIZER_DIR = "scratch_tokenizer"
MAX_INPUT = 256
MAX_OUTPUT = 128
EPOCHS = 3
BATCH_SIZE = 4
VOCAB_SIZE = 32000
# ===================================

# ========== STEP 1: Load and Combine Datasets ==========
dfs = [pd.read_csv(path)[["input_text", "output_text"]].dropna() for path in CSV_PATHS]
df = pd.concat(dfs, ignore_index=True)
dataset = Dataset.from_pandas(df)

# ========== STEP 2: Train Tokenizer (ONLY ONCE) ==========
if not os.path.exists(os.path.join(TOKENIZER_DIR, "tokenizer.json")):
    print("🧠 Training tokenizer from scratch...")

    from tokenizers import Tokenizer, models, trainers, pre_tokenizers, decoders, processors

    with open("combined.txt", "w", encoding="utf-8") as f:
        for row in df.itertuples():
            f.write(str(row.input_text) + "\n")
            f.write(str(row.output_text) + "\n")

    tokenizer_model = Tokenizer(models.BPE())
    tokenizer_model.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=True)
    tokenizer_model.decoder = decoders.ByteLevel()
    trainer = trainers.BpeTrainer(vocab_size=VOCAB_SIZE, special_tokens=["<pad>", "<s>", "</s>", "<unk>"])
    tokenizer_model.train(["combined.txt"], trainer)

    tokenizer_model.post_processor = processors.TemplateProcessing(
        single="<s> $A </s>",
        pair="<s> $A </s> </s> $B </s>",
        special_tokens=[
            ("<s>", tokenizer_model.token_to_id("<s>")),
            ("</s>", tokenizer_model.token_to_id("</s>"))
        ]
    )

    os.makedirs(TOKENIZER_DIR, exist_ok=True)
    tokenizer_model.save(os.path.join(TOKENIZER_DIR, "tokenizer.json"))

    hf_tokenizer = PreTrainedTokenizerFast(
        tokenizer_file=os.path.join(TOKENIZER_DIR, "tokenizer.json"),
        bos_token="<s>", eos_token="</s>",
        unk_token="<unk>", pad_token="<pad>"
    )
    hf_tokenizer.save_pretrained(TOKENIZER_DIR)

# ========== STEP 3: Load Tokenizer ==========
tokenizer = PreTrainedTokenizerFast.from_pretrained(TOKENIZER_DIR, model_max_length=MAX_INPUT)

# ========== STEP 4: Load or Initialize Model ==========
if PREV_MODEL_DIR and os.path.exists(os.path.join(PREV_MODEL_DIR, "pytorch_model.bin")):
    print(f"🔁 Loading previous model from: {PREV_MODEL_DIR}")
    model = T5ForConditionalGeneration.from_pretrained(PREV_MODEL_DIR)
else:
    print("🚀 Creating NEW model from scratch...")
    config = T5Config(
        vocab_size=VOCAB_SIZE,
        d_model=512,
        d_ff=2048,
        num_layers=6,
        num_heads=8,
        dropout_rate=0.1,
        eos_token_id=tokenizer.convert_tokens_to_ids("</s>"),
        pad_token_id=tokenizer.convert_tokens_to_ids("<pad>"),
        decoder_start_token_id=tokenizer.convert_tokens_to_ids("<pad>")
    )
    model = T5ForConditionalGeneration(config)

# ========== STEP 5: Tokenize Dataset ==========
def tokenize(example):
    input_enc = tokenizer(example["input_text"], truncation=True, padding="max_length", max_length=MAX_INPUT)
    target_enc = tokenizer(example["output_text"], truncation=True, padding="max_length", max_length=MAX_OUTPUT)
    input_enc["labels"] = target_enc["input_ids"]
    return input_enc

from datasets.utils.logging import set_verbosity_error
set_verbosity_error()

tokenized = dataset.map(tokenize, batched=True)
data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)

# ========== STEP 6: Train ==========
training_args = TrainingArguments(
    output_dir=MODEL_DIR,
    per_device_train_batch_size=BATCH_SIZE,
    num_train_epochs=EPOCHS,
    save_strategy="epoch",
    logging_steps=10,
    logging_dir="./logs",
    fp16=torch.cuda.is_available(),
    save_safetensors=False
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized,
    data_collator=data_collator
)

trainer.train()

# ========== STEP 7: Save Model ==========
print("💾 Saving model and tokenizer...")
try:
    model.save_pretrained(MODEL_DIR, safe_serialization=False)
    tokenizer.save_pretrained(TOKENIZER_DIR)
    print(f"✅ Model saved to: {MODEL_DIR}")
except Exception as e:
    print(f"⚠️ Failed to save model: {e}")

# ========== STEP 8: Evaluation (ROUGE-L) ==========
print("📊 Evaluating with ROUGE-L...")
rouge = evaluate.load("rouge")

predictions = []
references = []

for row in df.sample(n=min(50, len(df)), random_state=42).itertuples():
    input_ids = tokenizer(row.input_text, return_tensors="pt", truncation=True, padding="max_length", max_length=MAX_INPUT).input_ids
    with torch.no_grad():
        output_ids = model.generate(input_ids, max_length=MAX_OUTPUT, num_beams=4)
    decoded = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    predictions.append(decoded)
    references.append(row.output_text)

rouge_result = rouge.compute(predictions=predictions, references=references)
print("📈 ROUGE-L F1 Score:", round(rouge_result["rougeL"], 4))


🚀 Creating NEW model from scratch...


Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Passing a tuple of `past_key_values` is deprecated and will be removed in Transformers v4.48.0. You should pass an instance of `EncoderDecoderCache` instead, e.g. `past_key_values=EncoderDecoderCache.from_legacy_cache(past_key_values)`.


Step,Training Loss
10,2.226500
20,1.165800
30,0.672400


In [3]:
# ========== STEP 9: Sample Test ==========
def summarize_code(code_snippet: str):
    input_ids = tokenizer(code_snippet, return_tensors="pt", truncation=True, padding="max_length", max_length=MAX_INPUT).input_ids
    with torch.no_grad():
        summary_ids = model.generate(input_ids, max_length=MAX_OUTPUT, num_beams=4)
    return tokenizer.decode(summary_ids[0], skip_special_tokens=True)

print("\\n🧪 Running a sample test from the dataset:")
sample_row = df.iloc[0]
print("🔹 Input Code:\\n", sample_row.input_text)
print("🔹 Expected Summary:\\n", sample_row.output_text)
print("🔹 Model Prediction:\\n", summarize_code(sample_row.input_text))

\n🧪 Running a sample test from the dataset:
🔹 Input Code:\n import folium
map = folium.Map(location=[37.7749, -122.4194], zoom_start=13)
map.save('map.html')
🔹 Expected Summary:\n Creates a simple interactive map centered on San Francisco using Folium.
🔹 Model Prediction:\n Creates a simple interactive map centered on an Francisco using Folium.
